# Unsupervised Learning: Cluster analysis and dimension reduction.

This notebook has two parts covering unsupervised learning, using the same dataset.

Firstly it does a k-means clustering analysis, that that uses principal component analysis (PCA).

Second, it looks in more depth at the PCA.

## 1. Overview

Today we learned about about clustering for unsupervised machine learning and the k-means algorithm in particular. For a user-specified number $k$ of clusters, k-means assigns data points to their nearest centroid, which is iteratively readjusted to the mean of each cluster and points reassigned until clusters don't change anymore (or not much anymore). 


In this notebook we will use k-means clustering to analyse the Breast Cancer Wisconsin (Diagnostic) Data Set - check out the description here: https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data.

The data is in fact labelled - the column ```"diagnosis"``` takes values 'B' (benign) and 'M' (malignant). Here, we will ignore the outcome labels at first, perform clustering and then check if the assigned clusters make any sense, i.e. coincide with the observed outcomes. 

## 2. Import and data prep
### 2.1. Importing Dependencies

As usual, we start by importing libraries we will use later on. Throughout the notebook, if any functions are unclear, try googling the library and function to familiarize yourself with the functions and their in- and outputs. 

In [ ]:
import numpy as np 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn import preprocessing
from subprocess import check_output
from sklearn.metrics import confusion_matrix, accuracy_score, silhouette_score, adjusted_rand_score

### 2.2. Load data

**TASK:** Load the file ```breast-cancer-wisconsin.csv```, located in the Datasets directory.

In [ ]:
data = ...

### 2.3. Data summary

As usual, we start by having a look at the data. 

**TASK:** Use ```head()```, ```describe()``` and ```info()``` on the dataframe to get a first idea.

**TASK:** Check how many data points are classed as benign/malignant using ```groupby()``` and ```size()```, just as we did in the kNN notebook. 

**TASK:** Check for missing values (NaN = not a number) using ```isna()``` on the dataframe, just as we did in the regression notebook. 

**TASK:** We don't need the first and last column ('id', 'Unnamed: 32') - drop them. 

In [ ]:
# Cleaning and modifying the data


**TASK:** Assign the number of data points and columns to variables ```ndat, nvar``` using ```shape```.

Last week, we discussed that machine learning algorithms actually run on a matrix of numbers, so we need to map non-numerical variables to numbers in order to run these algorithms.  

So, finally, we map the diagnostic values to binary 0/1 values. 

In [ ]:
# Mapping Benign to 0 and Malignant to 1 
data['diagnosis'] = data['diagnosis'].map({'M':1,'B':0})

## 3. Exploratory Data Analysis

**TASK:** In the past two sessions, we've used a variety of plotting tools. Think which ones you found most useful and try them on this dataset. Keep in mind though, that this dataset has a lot more data columns than those we've used in the previous two sessions. This might make it a bit more cumbersome to look at all variables at once. 

Try an example visualization using a pair plot. 

**Caution:** Plotting all data columns using ```sns.pairplot()``` takes a long time (~2 minutes on a MacBook Air or on Noteable)! It also looks overwhelming. If you'd like to have a quick look, choose just up to 5-10 columns (0: "diagnosis" should be included if you'd like to use it for ```hue```).

**Reflect:** Do you think the data look well separable, i.e. a clustering method would be able to find clusters that roughly coincide with the diagnotic outcomes?

**TASK:** Compute and plot the correlation matrix, just as we did in the Class 12 homework notebook. Try to give this an informative title, and informative colourmap.

In [ ]:
corr_matrix = ...

sns.heatmap(...)


**Reflect**: what is this correlation heatmap telling you about the patterns in the data?

## 4. Data preparation
### 4.1. Data scaling

Data scaling is important, as we're assigning clusters based on Euclidean distance. This means that for all data variables to get the same importance, they need to range on the same scale. 

**TASK:** Scale the data using ```scale()``` from sklearn ```preprocessing``` library we imported at the beginning. Call the scaled data ```datas```, short for "data scaled". We don't need to rescale the ```"diagnosis"``` column, because we are not using this in the clustering.

In [ ]:
# Scaling the dataset
datas = pd.DataFrame(preprocessing.scale(data.iloc[:,1:nvar+1]))
datas.columns = list(data.iloc[:,1:nvar+1].columns)
datas['diagnosis'] = data['diagnosis']

**TASK:** Assign the scaled data to a variable ```X```, which we will use for the cluster analysis. Don't forget to drop diagnosis, which we are not using in the clustering. 

In [ ]:
# Creating the high dimensional feature space X
datas_nodiagnosis = ...
X = ...
# you might want to inspect X now ... take a look at it

### 4.2. Dimensionality reduction for later visualisation

We have 30 variables, and it is hard to visualize results in 30-D.

Here we use dimensionality reduction, to project data down to 2D and visualize clustering results later on.

#### 4.2.1. Principal component analysis

Principal component analysis (PCA) is a popular technique for analyzing large datasets containing a high number of dimensions/features per observation, increasing the interpretability of data while preserving the maximum amount of information, and enabling the visualization of multidimensional data.

Here we will apply PCA to the scaled data.

In [ ]:
from sklearn.decomposition import PCA

pca=PCA()
#Fit PCA to the dataset (only variables, excluding class)
pca.fit(datas_nodiagnosis)

# datapca is the data projected onto the principle components
datapca = pca.transform(datas_nodiagnosis)
outcomes = data.diagnosis

Next inspect the `datapca` object to see what it is. Be sure to check the shape and size of this, and compare it to the size of the data.

In fact, each **column** of this matrix is a principal component. We'll explain this more later.

In [ ]:
datapca.shape

In [ ]:
data.shape

In [ ]:
datapca

## 5. Cluster analysis

### 5.1. Perform k-means clustering

At last, we can use k-means to cluster our dataset. 

**TASK:** Create a k-means model object called ```km``` using ```KMeans()``` and then fit it using ```fit_predict()``` on the data vector ```X```. Use 2 clusters.

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(...)
kY = ...

In [ ]:
# Always inspect your objects! What is this telling you?
kY

### 5.2. Cluster visualisation


#### 5.2.1 A small example of cluster visualisation

First let's use just the first two variables in the dataset: radius_mean, and texture_mean. You already plotted these in the pairplot above.

**TASK:** Make two scatter plots of texture_mean against radius_mean. One should be coloured by the kNN cluster prediction `kY`, and the other by actual diagnosis `diagnosis

In [ ]:
sns.scatterplot(x = data["radius_mean"], ... )

In [ ]:
sns.scatterplot( ... )

Compare these - do the clusters correspond to the diagnosis?

However, we have 30 variables in the dataset, and this only views the clusters in 2 of those variables. It's time to use the PCA dimension reduction.

#### 5.2.2 Cluster visualisation and dimension reduction

Earlier in **§4** we computed the principal components. Use it now to visualize your clustering results in 2D. What does this 2D projection show?

**TASK:** Make two scatter plots using the first two principal components (```datapca```) computed in **§4.2.1** as axes. In the first scatter plot colour the data points by their cluster labels ```kY```, and in the second plot, for comparison, use the actual classes. 

**Reflect:** From the above plots, do you think the clustering did a good job? In which respects do the clusters resemble the true diagnosis, in which respects do they differ?

### 5.3. Evaluation

Since we know the actual outcomes, we can compare them to the assigned clusters. 


**TASK:** Compute the confusion matrix (like we did for kNN) and plot a heatmap of it (like we did in the regression notebook). 

In [ ]:
# The clustering doesn't know which class is which, so we might need to invert cluster indices (swap 0 and 1)
# kY = [int(x) for x in kY==0]

cm = ...
sns.heatmap(cm, annot=True)

accuracy = accuracy_score(data.diagnosis, kY)
print('\n Clusters coincide with diagnosis with ' + str(round(accuracy*100, 2)) + '% accuracy.')
cm

## 6. Finding a good $k$

Above, we used $k=2$, because we knew that there are two different diagnostic outcomes. In principle, however, when faced with a clustering problem, we don't *a priori* know the number of clusters. In the lecture, we learned that we can use an elbow plot to find the best $k$. 

**TASK:** Within a ```for``` loop, fit a k-means model for $k=1,...,10$ and compute the total within-cluster variation (**tip:** it can be accessed via the ```inertia_``` property of the fitted model). Store the within-cluster variation in an array.

In [ ]:
# wcss is short for "within-cluster-sum-of-squares"
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters = i, ...)
    kmeans.fit(X)
    wcss.append(...)

wcss

**TASK:** Plot this within-cluster sum of squares against the number of clusters, $k$, to get the elbow plot. What does the plot say about the optimal value of $k$?

In [ ]:
sns.lineplot(...)



## 6. Correlation and covariance data analysis

**TASK:** Check again the correlation matrix of the data, that we looked at above.

Now we could check out the covariance matrix of the data or its scaled version as we heard in the lecture that they have something to do with the PCA projection. How do the correlation matrix and the covariance matrix relate to each other?

In [ ]:
corr_matrix = ...

fig = ...

**TIP:** To calculate covariance, use the `cov` function from pandas.

In [ ]:
cov_matrix = ...

fig = ...

### 6.1. For the mathematically curious

Optional - Using the ```linalg``` library from numpy, compute the eigenvalues of the above covariance matrix. Create two bar plots of the unnormalized and normalized eigenvalues (i.e. divided by the sum of eigenvalues). We can check later if they coincide with the explained variances by the PC's.

**TIP:** convert the covariance matrix to an array first, before feeding it to the linalg library, using the function `to_numpy`.

In [ ]:
from numpy import linalg

eigenvalues, eigenvectors = linalg.eig(cov_matrix)

df_ev = pd.DataFrame({'Eigenvalues Covariance matrix': ...,
                      'Normalized eigenvalues': ...,
                      'Eigenvalue number' : ...})

plt.figure(figsize = (15,3))
sns.barplot(x = ..., y = 'Eigenvalues Covariance matrix', data = ...)

In [ ]:
plt.figure(figsize = (15,3))
sns.barplot(x = ..., y = 'Normalized eigenvalues', data = ...)

## 7. Principal Component Analysis (PCA)

Now we will use PCA to find a linear projection of the variable space. As above, but looking more in depth at what PCA is doing.

First, we will look at the full PCA space to then decide how to reduce it. 

### 7.1. Full PCA

**TASK:** Create a PCA object called ```pca``` using ```decomposition.PCA()```, then transform the data into the PCA space by using ```fit_transform``` on ```scaled_X```. Check out the transformed data.

This is the same as above.

In [ ]:
pca = ...
tdata = ...
tdata

**Reflect:** Have the dimensions been reduced? What is the new transformed data?

We can see above that the dimensions after PCA have not reduced. We've got entire set of new features projected onto the principal componenets. We have to find which or if all components are actualy important. 

### 7.2. Analysis of the PCA space

**TASK:** Check out the ```explained_variance_``` and ```explained_variance_ratio_``` attributes of the pca object. Create bar plots of them and compare to the bar plots of the eigenvalue

In [ ]:
#Information content by all new indep variables
...

**TASK:** Make a scree plot / elbow plot based on the ```explained_variance_ratio_```. Is there a clear scree point / elbow?

In [ ]:
df_expvar = pd.DataFrame({'% variation explained': ...,
                          'PCs': range(1,nvar)})

plt.figure(figsize = (15,3))
sns.barplot(x = 'PCs',y = '% variation explained',data = df_expvar);

**TASK:** Now plot the cumulative explained variance ratios using ```np.cumsum()```. Do you find this or the scree plot more informative?

In [ ]:
df_cexpvar = pd.DataFrame({'% variation explained': ...,
                    'Number of PCs': range(1,nvar)})

plt.figure(figsize = (25,6))
sns.barplot(x = 'Number of PCs',y = '% variation explained',data = df_cexpvar);

**Reflect:** Look at the scree and the cumulative plots. How many components are required to explain at least 80% of the variance in the data?

We can see that over 80% of information is explained by the first five components. 

**Optional TASK:** If you checked out the eigenvalues of the covariance matrix in the optional task in **§3.1**, now check out the ```explained_variance_``` attribute of our pca object. Make a plot and compare both this and the scree plot to those in **§3.1**. How do they relate to each other?

In [ ]:
df1 = pd.DataFrame({'Variation explained':pca.explained_variance_,
                    'PCs':['PC' + str(x) for x in range(1,nvar)]})

plt.figure(figsize = (25,6))
sns.barplot(x = 'PCs',y = 'Variation explained',data = df1);

### 7.4. Reduced PCA


Above, we learned that 5 components are required to explain at least 80% of the variance in our data. Let's now do dimension reduction by only looking at the first 5 components. 

**TASK:** Again using ```decomposition.PCA()``` now compute a PCA projection of the data setting ```n_components=5```. Then transform ```scaled_X``` into the reduced PCA space and store in ```pca_5var```.

In [ ]:
#we will do PCA with only 5 components now as they seem to provide 80% of the information.
pca_red = decomposition.PCA(n_components=5)
pca_5var = pca_red.fit_transform(scaled_X)

**TASK:** Check out ```explained_variance_``` and its sum to make sure we really explained at least 80% of our data variance.

In [ ]:
pca_red.explained_variance_

In [ ]:
np.sum(pca_red.explained_variance_ratio_)*100

We can see that over 80% information is obtained in first 5 components.

**TASK:** Create a dataframe ```X_5PCs``` from the transformed data using ```pd.DataFrame()``` on the transformed data, i.e. using ```columns=['PC1','PC2','PC3','PC4','PC5']``` of ```pca_5var```. Check it out using ```head()```.

In [ ]:
X_5PCs = ...

**TASK:** In the lecture, we learned that PCA makes a rotated projection of the data, where correlations have been removed. Check if that's true.

In [ ]:
X_5PCs.corr()

In [ ]:
sns.heatmap(...);


We can see that there is no correlation among the principal components which is very important. 

## 8. Classification of the breast cancer data

### 8.1. On the reduced PCA space

We will now train LogisticRegression on the data projected onto the 5 principal components.

**Reflect:** Why would we choose logistic regression on this data? 

Logistic regression is a classification method for binary outcome data.

**TASK:** Use ```train_test_split()``` on ```X_5PCs``` and ```y``` to create a test-train data split with ```test_size=0.2``` and setting ```random_state``` to any number of your choice.

In [ ]:
X_5PCs_train, X_5PCs_test, y_train, y_test = ...

**TASK:** Create a model called ```logreg``` using ```LogisticRegression()``` and setting ```solver='lbfgs'```. Fit the model on the training data using ```fit()``` and then make predictions on the test set using ```predict()``` on the logreg model. 

In [ ]:
# create logreg model
logreg = ...

# fit on training data


# predict on test data
y_pred_test = ...

**TASK:** Print the confusion matrix and the accuracy of the logreg model using ```confusion_matrix()``` and ```accuracy_score()``` on the test data.

In [ ]:
print(...)

### 8.2. On the original data space

Now let's see how the regression on the reduced PCA space compares to regression on the original data space. 

#### 8.2.1. Label encoding

Note, we mapped the diagnostic values to binary values above. Check this!

In [ ]:
data['diagnosis']

**TASK:** Assign the data to ```X```, excluding ```diagnosis```, and assign the latter to ```y```.

In [ ]:
X = data.loc[:, 'radius_mean':'fractal_dimension_worst']

y = data.diagnoses

**TASK:** Use ```train_test_split()``` on ```X``` and ```y``` to create a test-train data split with ```test_size=0.2``` and setting ```random_state``` to any number of your choice.

In [ ]:
#let's compare Logistic Regession without PCA when we have all of the original features
X_train, X_test, y_train, y_test = ...

**TASK:** Create a model called ```logreg2``` using ```LogisticRegression()``` and setting ```solver='lbfgs'```. Fit the model on the training data using ```fit()``` and then make predictions on the test set using ```predict()``` on the ```logreg2``` model. 

In [ ]:
# create logreg model
logreg2 = ...

# fit on training data

# predict on test data
y_pred_test2 = ...

**TASK:** Print the confusion matrix and the accuracy of the logreg2 model using ```confusion_matrix()``` and ```accuracy_score()``` on the test data.

In [ ]:
print(...)

**TASK:** Compare the accuracy we got for classification when using the original data and when it's been projected onto the reduced PCA space. 

Accuracy is already high without PCA, but we can see that we gain accuracy when PCA is used compared to when it is not used. Since correlation has been removed from the data, the classifier has an easier task. 

**Optional TASK:** Can we get even better classification results by using more principal components?

### 8.3. Visualizing the classification results

Let's visualize the classification results on the whole data set.

**TASK:** First make predictions from the logreg models fitted on the reduced PCA and the original space, i.e. ```logreg``` and ```logreg2```, this time on the whole dataset, i.e. on ```X_5PCs``` and ```X```, respectively.

In [ ]:
y_pred_all = ...
y_pred_all2 = ...

**TASK:** Make 3 scatter plots of the data projected onto the first two components, i.e. plotting ```X_5PCs.PC1``` vs ```X_5PCs.PC2```, and colour the points by 
1. Predictions from classification on the reduced PCA space.
2. Predictions from classification on the original data space.
3. True diagnoses.

In [ ]:
# this line allows us to make 3 plots next to each other
# and adjusts figure size to work for 3 plots
# look up the matplotlib library for subplots to learn more
f, axes = plt.subplots(1, 3, sharey=True, figsize = (16,5))

# create chart in first subplot axes[0]
sns.scatterplot(..., ax=axes[0])
axes[0].set_title('...')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

# create chart in second subplot axes[1]
sns.scatterplot(..., ax=axes[1])
...

# create chart in third subplot axes[2]
sns.scatterplot(..., ax=axes[2])
...


Since accuracy is quite high for both classifiers, both predictors are very close to the true diagnoses. We know however that we main some accuracy gains, and we can keep in mind next time we do classification and achieve low accuracy that PCA or dimension reduction, more generally, can help us improve predictions. 

This notebook has beed adapted from: https://www.kaggle.com/code/vishwaparekh/cluster-analysis-of-breast-cancer-dataset/notebook